# Distributed HDP-GMM five-seed metrics

This notebook runs the Ray-based distributed Gibbs sampler for the HDP-GMM over five random seeds and summarizes ACC, damage detection accuracy, Brier score, expected calibration error, and runtime.


In [1]:
import numpy as np
import ray
from copy import deepcopy
import matplotlib.pyplot as plt
import scipy.special as ssp

from hdpgmm_syn import Gaussian, WorkerLevelGibbsSampler
from hdpgmm import GibbsSampler
from model_loglikelihood import dict2mix, all_loglike
import time
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pandas as pd
import torch
from scipy.optimize import linear_sum_assignment
from typing import List, Callable, Union, Any, TypeVar, Tuple
Tensor = TypeVar('torch.tensor')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib

from collections import Counter

seed = 42
torch.manual_seed(seed)  # ensure reproducible results
np.random.seed(seed)


e:\Anaconda\envs\vscode\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-21 16:35:57,531	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


# Dataset


In [ ]:
'''use a numerical dataset to test the model
label  damaged_floor     damage_extent         number_of_samples
0           0            0%                     1500
1           1            3%                     500
2           1            6%                     500
3           1            10%                    500
4           1,3          5%,10%                 500
5           1,3,5        5%,10%,15%             500
6           2,4,6        10%,15%,20%            500
7           1,3,5,7      10%,15%,20%,25%        500


'''
# Read data from CSV
features = pd.read_csv("TF_mag_numerical_8class_1000features_20dB.csv")
features = features.astype("float32")
# # # Convert DataFrame to PyTorch tensors
X = torch.tensor(features.values[:,:])
X = X.t()

input_dim = X.shape[1]
print(X.shape)


a = []
for i in range(1500):
    a.append(0)
for i in np.arange(1500,2000):
    a.append(1)
for i in np.arange(2000,2500):
    a.append(2)
for i in np.arange(2500,3000):
    a.append(3)
for i in np.arange(3000,3500):
    a.append(4)
for i in np.arange(3500,4000):
    a.append(5)
for i in np.arange(4000,4500):
    a.append(6)
for i in np.arange(4500,5000):
    a.append(7)
print(len(a))

y0 = torch.tensor(a)
y0 = y0.unsqueeze(1)
y = [int(_) for _ in y0]
y = torch.tensor(y)
num_classes = y.max().item() + 1
print(f"number of total classes: {num_classes}")

plt.plot(y)
plt.show()

# Distributed sampler helpers


In [3]:
# subtract one worker's contribution from the global state
def subtract_worker_contribution(global_state, worker_local_state, dim):

    assert len(global_state.keys()) >= len(worker_local_state.keys()), "Worker local state has more components than global state."
    
    """Return global_state with one worker's cached contribution removed."""
    adjusted_state = {}

    for comp_id, stats in global_state.items():
        key = int(comp_id)
        adjusted_state[key] = {
            'nk': float(stats.get('nk', 0.0)),
            'sum_x': np.asarray(stats.get('sum_x')).reshape(1, dim).copy(),
            'sum_xx': np.asarray(stats.get('sum_xx')).reshape(dim, dim).copy(),
            'm_k': float(stats.get('m_k', 0.0))
        }


    stats_map = worker_local_state.get('component_stats') if isinstance(worker_local_state, dict) and 'component_stats' in worker_local_state else worker_local_state

    assert len(adjusted_state.keys()) >= len(stats_map.keys()), "adjusted_state is the copy of global_state, stats_map is the copy of local state."

    comp_id = 0


    for local_id, stats in stats_map.items():
        
        if int(local_id) in adjusted_state:
   
            adjusted_state[int(local_id)]['nk'] -= float(stats.get('nk', 0.0))
            adjusted_state[int(local_id)]['sum_x'] -= np.asarray(stats.get('sum_x', np.zeros((1, dim)))).reshape(1, dim)
            adjusted_state[int(local_id)]['sum_xx'] -= np.asarray(stats.get('sum_xx', np.zeros((dim, dim)))).reshape(dim, dim)
            adjusted_state[int(local_id)]['m_k'] -= float(stats.get('m_k', 0.0))

    return adjusted_state
    
# for mk, we only need the local mk, so we need to subtract mk_(-p), i.e., the effective global mk input to the worker
# note that mk is actually a global stats
def compute_local_component_stats(sampler):
    dim = sampler._corpus[0].shape[1]
    component_stats = {}
    doc_stats = []
    for d in range(len(sampler._corpus)):
        doc = sampler._corpus[d]
        tables = sampler._t_dv[d]
        topics = sampler._k_dt[d]
        counts = sampler._n_dt[d]
        active_tables = np.where(counts > 0)[0]
        doc_stats.append({'n': int(doc.shape[0]), 'm': int(len(active_tables))})

    for idx in sampler.params_ordered.keys():
        component_stats[int(idx)] = {}
        if hasattr(sampler.params_ordered[idx],'ss'):
            component_stats[int(idx)]['nk'] = sampler.params_ordered[idx].ss['nk']
            component_stats[int(idx)]['sum_x'] = sampler.params_ordered[idx].ss['sum_x']
            component_stats[int(idx)]['sum_xx'] = sampler.params_ordered[idx].ss['sum_xx']
            
        else:
            component_stats[int(idx)]['nk'] = 0.0
            component_stats[int(idx)]['sum_x'] = np.zeros(dim)
            component_stats[int(idx)]['sum_xx'] = np.zeros((dim, dim))
        
        component_stats[int(idx)]['m_k'] = sampler.params_ordered[idx].mk - sampler.params_ordered[idx].prior_mk # effective local mk

        if len(sampler.params_ordered.keys()) != sampler._m_k.shape[0]:
            print('mk', sampler._m_k)
            
            for idx in sampler.params_ordered.keys():
                print('comps', idx)
                print('nk',sampler.params_ordered[idx].nk)

        assert(len(sampler.params_ordered.keys()) == sampler._m_k.shape[0]), ('mismatch between component number and mk')
        
    return component_stats, doc_stats


In [4]:
def split_docs_across_workers(data, num_workers):
    shards = np.array_split(data, num_workers, axis=0)
    data_shards = [np.array(shard) for shard in shards if len(shard) > 0]
    num_docs = data.shape[0]
    doc_index_splits = [split.astype(int) for split in np.array_split(np.arange(num_docs), num_workers) if len(split) > 0]
  
    # assert len(data_shards) == num_workers
    assert len(data_shards) == len(doc_index_splits)
    # shards = {worker_id: data[data assigned to worker_id]}
    return data_shards, doc_index_splits

# build initial global state from entire dataset, 
# each document is initialized as a component
def build_initial_global_state(data):
    assert len(data.shape) == 3
    dim = data.shape[2]
    state = {}
    for idx, doc in enumerate(data):
        nk = float(doc.shape[0])
        sum_x = doc.sum(axis=0, keepdims=True)
        sum_xx = doc.T @ doc
        state[idx] = {
            'nk': nk,
            'sum_x': sum_x,
            'sum_xx': sum_xx,
            'm_k': 1.0
        }
    return state

# create local states for each worker
def build_initial_local_states(data_shards, doc_index_splits, initial_global_state):
    initial_local_states = {}
    worker_id = 0
    for doc_ids in doc_index_splits:
        worker_state = {
            int(comp_id): {
                'nk': float(stats['nk']),
                'sum_x': np.asarray(stats['sum_x']).copy(),
                'sum_xx': np.asarray(stats['sum_xx']).copy(),
                'm_k': float(stats['m_k'])
            }
            for comp_id, stats in initial_global_state.items()
        }
        for doc_idx in doc_ids:
            key = int(doc_idx)
            if key not in worker_state:
                continue
            doc_data = data[key]
            if doc_data.size == 0:
                continue
            stats = worker_state[key]
            doc_num = float(doc_data.shape[0])
            doc_sum_x = doc_data.sum(axis=0, keepdims=True)
            doc_sum_xx = doc_data.T @ doc_data

            stats['nk'] =stats['nk'] - doc_num
            stats['sum_x'] = stats['sum_x'] - doc_sum_x
            stats['sum_xx'] = stats['sum_xx'] - doc_sum_xx
            stats['m_k'] = stats['m_k'] - 1.0

            assert worker_state[key]['nk'] >= 0
            assert worker_state[key]['m_k'] >= 0

        initial_local_states[worker_id] = worker_state
        worker_id += 1

    return initial_local_states

In [5]:
def merge_worker_outputs(worker_outputs, dim):
    global_state = {}

    for payload in worker_outputs:

        for comp_id, stats in payload.items():

            if comp_id not in global_state:
                nk = float(stats['nk'])
                # if nk < 0:
                #     continue
                sum_x = np.asarray(stats['sum_x']).reshape(1, dim).copy()
                sum_xx = np.asarray(stats['sum_xx']).reshape(dim, dim).copy()

                global_state[comp_id] = {
                    'nk': nk,
                    'sum_x': sum_x,
                    'sum_xx': sum_xx,
                    'm_k': float(stats['m_k'])
                }

            else:
                global_state[comp_id]['nk'] += float(stats['nk'])
                global_state[comp_id]['sum_x'] += np.asarray(stats['sum_x']).reshape(1, dim).copy()
                global_state[comp_id]['sum_xx'] += np.asarray(stats['sum_xx']).reshape(dim, dim).copy()
                global_state[comp_id]['m_k'] += float(stats['m_k'])
        
    return global_state


In [6]:
def sample_concentration_params(data, global_m_k, alpha_a=100., alpha_b=100., gamma_a=100., gamma_b=100., max_iter_alpha=None, max_iter_gamma=None):
    '''procedure of sampling alpha:
    sample alpha from Gamma(.)
    sample w_j from Beta(.)
    sample s_j from Bernoulli(.)'''

    D = data.shape[0] # number of documents
    n_j = [data[j].shape[0] for j in range(D)] # number of data points in each document
    K = len(global_m_k) # number of topics

    alpha = np.random.gamma(alpha_a, alpha_b)
    gamma = np.random.gamma(gamma_a, gamma_b)
    iter_alpha = 0
    iter_gamma = 0
    if max_iter_alpha and max_iter_gamma:
        for iter_alpha in range(max_iter_alpha):
            alpha_prev = alpha
            w_alpha = np.array([np.random.beta(alpha+1, n_j[j]) for j in range(D)]) # array of all w_j
            s_alpha = np.array([np.random.binomial(1, np.sum(n_j[j]) / (np.sum(n_j[j]) + alpha)) for j in range(D)])
            # sample alpha from the gamma distribution
            #rate = 1./self._alpha_b - np.sum(np.log(self._w_alpha))
            rate = (alpha_b - np.sum(np.log(w_alpha)))
            alpha = np.random.gamma(alpha_a + np.sum(global_m_k) - np.sum(s_alpha), 1./rate)
            #print('expected alpha:', (alpha_a + np.sum(global_m_k) - np.sum(s_alpha))* (1./rate))
        for iter_gamma in range(max_iter_gamma):
            gamma_prev = gamma
            w_gamma = np.random.beta(gamma+1, np.sum(global_m_k))
            pi = gamma_a + K - 1
            # np.random.binomial(n,p), draw n samples from the binomial distribution, P(X=1)=p, P(X=0)=1-p, select a Gamma density as the posterior of gamma is a mixture of two Gamma densities
            s_gamma = np.random.binomial(1, (gamma_b - np.log(w_gamma)) * np.sum(global_m_k) / (
            pi + (gamma_b - np.log(w_gamma)) * np.sum(global_m_k))) 
            # rate = 1./self._gamma_b - np.log(self._w_gamma)
            rate = (gamma_b - np.log(w_gamma))
            # in np.random.gamma, it uses the shape闁炽儲鎮瀋ale parameterization, i.e., np.random.gamma(shape, scale); in Escobar闁炽儲褰瀍st, it uses the shape闁炽儲鎮渁te parameterization, so here we use scale = 1./rate
            gamma = np.random.gamma(gamma_a + K - s_gamma, 1./rate)
    else:
        while True:
            alpha_prev = alpha
            w_alpha = np.array([np.random.beta(alpha+1, n_j[j]) for j in range(D)]) # array of all w_j
            s_alpha = np.array([np.random.binomial(1, np.sum(n_j[j]) / (np.sum(n_j[j]) + alpha)) for j in range(D)])
            # sample alpha from the gamma distribution
            #rate = 1./self._alpha_b - np.sum(np.log(self._w_alpha))
            rate = (alpha_b - np.sum(np.log(w_alpha)))
            alpha = np.random.gamma(alpha_a + np.sum(global_m_k) - np.sum(s_alpha), 1./rate)
            #print('expected alpha:', (alpha_a + np.sum(global_m_k) - np.sum(s_alpha))* (1./rate))

            if abs(alpha-alpha_prev)/alpha < 1e-3:
                break
            iter_alpha += 1
        # sample gamma from a gamma distribution
        while True:
            gamma_prev = gamma
            w_gamma = np.random.beta(gamma+1, np.sum(global_m_k))
            pi = gamma_a + K - 1
            # np.random.binomial(n,p), draw n samples from the binomial distribution, P(X=1)=p, P(X=0)=1-p, select a Gamma density as the posterior of gamma is a mixture of two Gamma densities
            s_gamma = np.random.binomial(1, (gamma_b - np.log(w_gamma)) * np.sum(global_m_k) / (
            pi + (gamma_b - np.log(w_gamma)) * np.sum(global_m_k))) 
            # rate = 1./self._gamma_b - np.log(self._w_gamma)
            rate = (gamma_b - np.log(w_gamma))
            # in np.random.gamma, it uses the shape闁炽儲鎮瀋ale parameterization, i.e., np.random.gamma(shape, scale); in Escobar闁炽儲褰瀍st, it uses the shape闁炽儲鎮渁te parameterization, so here we use scale = 1./rate
            gamma = np.random.gamma(gamma_a + K - s_gamma, 1./rate)

            if abs(gamma - gamma_prev)/gamma < 1e-3:
                break
            iter_gamma += 1
        #print('alpha sampled in %i iterations, gamma sampled in %i iterations' % (iter_alpha, iter_gamma))
    return alpha, gamma


In [7]:
@ray.remote
class GibbsWorker:
    def __init__(self, worker_id, data_block, seed=None):
        self.worker_id = worker_id
        self.data = data_block
        self.dim = data_block.shape[2]
        self.seed = seed
        self._last_local_stats = {}
        self._sampler = None
        if self.seed is not None:
            np.random.seed(int(self.seed))
            torch.manual_seed(int(self.seed))
    
    def get_last_component_stats(self):
        return self._last_local_stats
    
    def get_sampler(self):
        return self._sampler

    def run_local_sampler(self, global_state, hyperparameter, iterations, snapshot_interval=50, compute_loglik=False):
        if self._sampler is None:
            self._sampler = WorkerLevelGibbsSampler(snapshot_interval=snapshot_interval, compute_loglik=compute_loglik)
            self._sampler.initialize(data=self.data, global_state=global_state, hyperparameter=hyperparameter)
        else:
            effective_global = subtract_worker_contribution(global_state, self._last_local_stats, self.dim)
            if hasattr(self._sampler, "set_global_snapshot"):
                self._sampler.set_global_snapshot(effective_global, hyperparameter)
            else:
                raise RuntimeError("Sampler must implement set_global_snapshot(...).")

        self._sampler.sample(iterations)
        component_stats, doc_stats = compute_local_component_stats(self._sampler)

        self._last_local_stats = {
            int(k): {
                'nk': float(v['nk']),
                'sum_x': np.asarray(v['sum_x']).copy(),
                'sum_xx': np.asarray(v['sum_xx']).copy(),
                'm_k': float(v['m_k']),
            }
            for k, v in component_stats.items()
        }
        return self._last_local_stats

In [8]:
def compute_global_loglik(data, global_components, current_state, samplers, hyperparameter, hyperprior=None):
    from scipy.stats import multivariate_normal
    num_comp = len(current_state)
        
    assert num_comp == len(global_components.keys()), "Mismatch between current_state and global_components."
    # pi_k = (N_jk+alpha*(mk/m+gamma))/(sum(alpha*(mk/m+gamma))+N_j)
    # N_jk: n_kd_total
    n_kd_total = np.empty((num_comp,0))
    for sampler in samplers:
        if sampler._n_kd.shape[0] < num_comp:
            zero_comps = num_comp - sampler._n_kd.shape[0]
            n_kd = np.concatenate([sampler._n_kd, np.zeros((zero_comps, sampler._n_kd.shape[1]))], axis=0)
        else:
            n_kd = sampler._n_kd
        n_kd_total = np.concatenate([n_kd_total, n_kd], axis=1) # shape: (num_comp, num_documents)
    
    # m_k
    mk = np.array([current_state[comp_id]['m_k'] for comp_id in range(num_comp)])
    m = np.sum(mk)

    # alpha, gamma
    alpha = hyperparameter['alpha']
    gamma = hyperparameter['gamma']

    D = n_kd_total.shape[1]
    loglik = np.zeros(D)
    dists = {}
    for k in range(num_comp):
        dists[k] = multivariate_normal(mean=global_components[k].mean.A1, cov=global_components[k].covar)
    for d in range(D):
        n_kd = n_kd_total[:, d]
        n_d  = float(n_kd.sum())

        # CRF closed-menu weights (ignoring "new" component); if you want "new", switch to denom = n_d + alpha etc.
        numer = n_kd + alpha * (mk / (m + gamma))
        denom = n_d + alpha * (m / (m + gamma))
        pi_k  = numer / denom  # shape (K,)

        s = 0.0
        for x in data[d]:
            
            # log_fk = np.array([global_components[k].logpdf(x) for k in range(num_comp)], dtype=float)
            log_fk = np.array([dists[k].logpdf(x) for k in range(num_comp)], dtype=float)
            terms  = np.log(pi_k) + log_fk
            terms  = np.log(pi_k) + log_fk
            s += ssp.logsumexp(terms)
        loglik[d] = s

    loglik_sum = np.sum(loglik)

    if hyperprior:
        loglik_sum += (hyperprior['alpha_a'] - 1)*np.log(hyperparameter['alpha']) - hyperparameter['alpha']/hyperprior['alpha_b'] - hyperprior['alpha_a']*np.log(hyperprior['alpha_b']) - ssp.gammaln(hyperprior['alpha_a'])
        loglik_sum += (hyperprior['gamma_a'] - 1)*np.log(hyperparameter['gamma']) - hyperparameter['gamma']/hyperprior['gamma_b'] - hyperprior['gamma_a']*np.log(hyperprior['gamma_b']) - ssp.gammaln(hyperprior['gamma_a'])
    
    return loglik_sum


In [9]:
def run_distributed_hdp(
    data_array,
    num_workers=3,
    local_iterations=1,
    global_iterations=5,
    snapshot_interval=1,
    compute_loglik=True,
    sample_concen=True,
    hyperparameter=None,
    hyperprior=None,
    base_seed=None,
):
    start_time = time.time()
    runningtime_snapshot = []

    if base_seed is not None:
        np.random.seed(int(base_seed))
        torch.manual_seed(int(base_seed))

    dim = data_array.shape[2]
    global_marginal_loglikelihoods = []

    if hyperprior is None:
        hyperprior = {'alpha_a': 100.0, 'alpha_b': 100.0, 'gamma_a': 100.0, 'gamma_b': 100.0}

    if hyperparameter is None:
        hyperparameter = {
            'alpha': 1.0,
            'gamma': 1.0,
            'mu_0': np.zeros((1, dim)),
            'kappa_0': 1.0,
            'nu_0': float(2),
            'Psi_0': np.eye(dim),
        }

    data_shards, doc_index_splits = split_docs_across_workers(data_array, num_workers)
    initial_global_state = build_initial_global_state(data_array)
    initial_local_states = build_initial_local_states(data_shards, doc_index_splits, initial_global_state)
    global_state = deepcopy(initial_global_state)

    assert len(doc_index_splits) == len(data_shards), "Mismatch between shard data and index tracking."

    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True, include_dashboard=False)

    worker_seeds = [None if base_seed is None else int(base_seed) + 1009 * (worker_id + 1) for worker_id in range(len(data_shards))]
    workers = [GibbsWorker.remote(worker_id, shard, worker_seeds[worker_id]) for worker_id, shard in enumerate(data_shards)]
    history = []

    for sync_iter in range(global_iterations):
        print('Global iteration %i' % sync_iter)

        if sample_concen:
            global_m_k = np.array([stats['m_k'] for comp_id, stats in global_state.items()])
            alpha, gamma = sample_concentration_params(
                data_array,
                global_m_k,
                hyperprior['alpha_a'],
                hyperprior['alpha_b'],
                hyperprior['gamma_a'],
                hyperprior['gamma_b'],
                max_iter_alpha=20,
                max_iter_gamma=20,
            )
            hyperparameter['alpha'] = alpha
            hyperparameter['gamma'] = gamma

        print('alpha:', hyperparameter['alpha'])
        print('gamma:', hyperparameter['gamma'])

        if sync_iter == 0:
            tasks = [
                worker.run_local_sampler.remote(
                    global_state=initial_local_states[worker_id],
                    hyperparameter=hyperparameter,
                    iterations=local_iterations,
                    snapshot_interval=snapshot_interval,
                    compute_loglik=compute_loglik,
                )
                for worker_id, worker in enumerate(workers)
            ]
        else:
            tasks = [
                worker.run_local_sampler.remote(
                    global_state=global_state,
                    hyperparameter=hyperparameter,
                    iterations=local_iterations,
                    snapshot_interval=snapshot_interval,
                    compute_loglik=compute_loglik,
                )
                for worker_id, worker in enumerate(workers)
            ]

        worker_outputs = ray.get(tasks)
        global_state = merge_worker_outputs(worker_outputs, dim)
        current_hyper = deepcopy(hyperparameter)
        current_state = deepcopy(global_state)

        history.append({
            'iteration': sync_iter,
            'num_components': len(current_state),
            'alpha': current_hyper['alpha'],
            'gamma': current_hyper['gamma'],
        })

        if (sync_iter + 1) % snapshot_interval == 0 or sync_iter == global_iterations - 1:
            runningtime_snapshot.append(time.time() - start_time)

        if compute_loglik:
            K = len(current_state)
            component_data = {k: np.array([]) for k in range(K)}
            samplers = ray.get([w.get_sampler.remote() for w in workers])
            for sampler in samplers:
                for component_id, param in sampler.params_ordered.items():
                    if hasattr(param, '_X'):
                        component_id = int(component_id)
                        component_data[component_id] = (
                            np.concatenate([component_data[component_id], param._X], axis=0)
                            if component_data[component_id].size
                            else param._X
                        )

            global_components = {}
            for comp_id in range(K):
                X_comp = component_data[comp_id] if component_data[comp_id].shape[0] > 0 else np.zeros((0, dim))
                global_components[comp_id] = Gaussian(
                    X=X_comp,
                    kappa_0=hyperparameter['kappa_0'],
                    nu_0=hyperparameter['nu_0'],
                    mu_0=hyperparameter['mu_0'],
                    Psi_0=hyperparameter['Psi_0'],
                )

            global_marginal_loglik = compute_global_loglik(
                data_array,
                global_components,
                current_state,
                samplers,
                hyperparameter,
                hyperprior=None,
            )
            global_marginal_loglikelihoods.append(global_marginal_loglik)
            print(
                f"Global iter {sync_iter:02d}: K={len(current_state)}, "
                f"alpha={current_hyper['alpha']:.3f}, gamma={current_hyper['gamma']:.3f}, "
                f"Global Marginal LogLik={global_marginal_loglik:.3f}"
            )
        else:
            print(f"Global iter {sync_iter:02d}: K={len(current_state)}, alpha={current_hyper['alpha']:.3f}, gamma={current_hyper['gamma']:.3f}")

    return current_state, current_hyper, history, workers, global_marginal_loglikelihoods, runningtime_snapshot


# PCA


In [10]:
pca_components = 10
samples_per_segment = 100

pca = PCA(n_components=pca_components)
data_pca = pca.fit_transform(X).astype("float32")
data_pca = data_pca.reshape(-1, samples_per_segment, pca_components)

data_raw = X.numpy().reshape(-1, 100, input_dim)
print(data_raw.shape)
print(data_pca.shape)

(50, 100, 1000)
(50, 100, 10)


# Metric helpers


In [11]:
def unsupervised_clustering_accuracy(y: Union[np.ndarray, torch.Tensor], y_pred: Union[np.ndarray, torch.Tensor]) -> tuple:
        """Unsupervised Clustering Accuracy
        """
        assert len(y_pred) == len(y)
        u = np.unique(y)
        n_true_clusters = len(u)
        v = np.unique(y_pred)
        n_pred_clusters = len(v)
        map_u = dict(zip(u, range(n_true_clusters)))
        map_v = dict(zip(v, range(n_pred_clusters)))
        inv_map_u = {v: k for k, v in map_u.items()}
        inv_map_v = {v: k for k, v in map_v.items()}
        r = np.zeros((n_pred_clusters, n_true_clusters), dtype=np.int64)
        for y_pred_, y_ in zip(y_pred, y):
            if y_ in map_u:
                r[map_v[y_pred_], map_u[y_]] += 1
        reward_matrix  = np.concatenate((r, r, r), axis=1)
        cost_matrix = reward_matrix.max() - reward_matrix
        row_assign, col_assign = linear_sum_assignment(cost_matrix)

        # Construct optimal assignments matrix
        row_assign = row_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
        col_assign = col_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
        assignments = np.concatenate((row_assign, col_assign), axis=1)
        assignments = [[inv_map_v[x], inv_map_u[y%n_true_clusters]] for x, y in assignments]

        optimal_reward = reward_matrix[row_assign, col_assign].sum() * 1.0
        return optimal_reward / y_pred.size, assignments  

def damage_detection_accuracy(k_dv_global, y_true, healthy_label=0, healthy_reference_fraction=0.8, min_count=10):
    """Damage detection accuracy that remains valid after sample shuffling.

    Healthy clusters are estimated from the first `healthy_reference_fraction`
    of samples whose true label is `healthy_label`, instead of assuming that
    healthy samples occupy the first block of the dataset.
    """
    k_dv_global_int = np.asarray(k_dv_global).astype(int)
    y_true_int = np.asarray(y_true).astype(int)

    if len(k_dv_global_int) != len(y_true_int):
        raise ValueError("k_dv_global and y_true must have the same length.")

    healthy_indices = np.flatnonzero(y_true_int == healthy_label)
    if len(healthy_indices) == 0:
        raise ValueError(f"No samples found for healthy_label={healthy_label}.")

    healthy_reference_end = max(1, int(len(healthy_indices) * healthy_reference_fraction))
    healthy_reference_indices = healthy_indices[:healthy_reference_end]

    health_labels_pre = np.unique(k_dv_global_int[healthy_reference_indices])
    counts = Counter(k_dv_global_int[healthy_reference_indices])
    health_labels = [label for label in health_labels_pre if counts[label] >= min_count]

    healthy_mask = y_true_int == healthy_label
    predicted_healthy_mask = np.isin(k_dv_global_int, health_labels)

    fp = int(np.sum(healthy_mask & ~predicted_healthy_mask))
    fn = int(np.sum(~healthy_mask & predicted_healthy_mask))
    dda = 1 - (fp + fn) / len(k_dv_global_int)

    return dda, fp, fn, health_labels


def brier_score(p_hat, y_true, classes=None, sample_weight=None):
    p_hat = np.asarray(p_hat, dtype=float)
    N, C = p_hat.shape
    if classes is None:
        classes = np.unique(y_true)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    Y = np.eye(C)[y_idx]
    if sample_weight is None:
        return np.mean(np.sum((p_hat - Y) ** 2, axis=1))
    w = np.asarray(sample_weight, dtype=float)
    w /= w.sum()
    return np.sum(w * np.sum((p_hat - Y) ** 2, axis=1))


def brier_by_class(p_hat, y_true, classes=None):
    p_hat = np.asarray(p_hat, dtype=float)
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    Y = np.eye(C)[y_idx]
    out = {}
    for c, cl in enumerate(classes):
        mask = y_idx == c
        out[cl] = np.mean(np.sum((p_hat[mask] - Y[mask]) ** 2, axis=1)) if mask.any() else np.nan
    return out


def brier_skill_score(p_hat, y_true, classes=None):
    if classes is None:
        classes = np.unique(y_true)
    counts = np.array([(y_true == c).sum() for c in classes], dtype=float)
    p_ref = counts / counts.sum()
    P_ref = np.tile(p_ref, (len(y_true), 1))
    BS = brier_score(p_hat, y_true, classes)
    BS_ref = brier_score(P_ref, y_true, classes)
    return 1.0 - (BS / BS_ref if BS_ref > 0 else np.nan)


def predict_cluster_posteriors_CRF(data_groups, global_components, n_kd_total, mk, alpha, gamma, ignore_new=True):
    K, D_docs = n_kd_total.shape
    m = float(np.sum(mk))
    R_list, idx_map = [], []

    for d in range(D_docs):
        n_kd = n_kd_total[:, d].astype(float)
        n_d = float(n_kd.sum())
        numer = n_kd + alpha * (mk / (m + gamma))
        denom = n_d + alpha * (m / (m + gamma))
        pi_k = numer / denom

        Xd = np.asarray(data_groups[d])
        R_d = np.empty((Xd.shape[0], K), dtype=float)
        log_pi = np.log(pi_k + 1e-300)
        for i, x in enumerate(Xd):
            log_fk = np.array([global_components[k].logpdf(x).item() for k in range(K)], dtype=float)
            z = log_pi + log_fk
            z -= ssp.logsumexp(z)
            R_d[i, :] = np.exp(z)
            idx_map.append((d, i))
        R_list.append(R_d)

    return R_list, idx_map


def estimate_Q_c_given_k(R, y_true, classes=None, reg=1e-6):
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    N, K = R.shape
    counts = np.zeros((C, K), dtype=float)
    for i in range(N):
        counts[y_idx[i], :] += R[i, :]
    counts += reg
    Q = counts / counts.sum(axis=0, keepdims=True)
    return Q, classes


def class_probs_from_R(R, Q):
    return R @ Q.T


def brier_from_hdp_state(data_groups, global_components, n_kd_total, mk, alpha, gamma, y_true, labeled_mask=None, classes=None):
    R_list, idx_map = predict_cluster_posteriors_CRF(
        data_groups, global_components, n_kd_total, mk, alpha, gamma, ignore_new=True
    )
    R = np.vstack(R_list)

    if labeled_mask is None:
        R_lab = R
        y_lab = y_true
    else:
        labeled_mask = np.asarray(labeled_mask, dtype=bool)
        R_lab = R[labeled_mask]
        y_lab = np.asarray(y_true)[labeled_mask]

    Q, classes = estimate_Q_c_given_k(R_lab, y_lab, classes=classes, reg=1e-6)
    P_hat = class_probs_from_R(R, Q)

    BS = brier_score(P_hat, y_true, classes=classes)
    BSc = brier_by_class(P_hat, y_true, classes=classes)
    BSS = brier_skill_score(P_hat, y_true, classes=classes)
    return BS, BSc, BSS, P_hat, R, Q


def learn_Q_soft(R, y_true, classes=None, reg=1e-6):
    if classes is None:
        classes = np.unique(y_true)
    C, K = len(classes), R.shape[1]
    cls2idx = {c: i for i, c in enumerate(classes)}
    y_idx = np.array([cls2idx[y] for y in y_true], int)
    E = np.zeros((C, K), float)
    for i in range(R.shape[0]):
        E[y_idx[i]] += R[i]
    Q = (E + reg) / (E + reg).sum(axis=0, keepdims=True)
    return Q, classes


def ece_toplabel(P_hat, y_true, n_bins=15):
    N, C = P_hat.shape
    conf = P_hat.max(axis=1)
    pred = P_hat.argmax(axis=1)
    correct = (pred == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if not np.any(m):
            continue
        ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece


def ece_ovr(P_hat, y_true, n_bins=15, classes=None):
    P_hat = np.asarray(P_hat, float)
    N, C = P_hat.shape
    if classes is None:
        classes = np.arange(C)
    cls2idx = {c: i for i, c in enumerate(classes)}
    y_idx = np.array([cls2idx[y] for y in y_true], int)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for c in range(C):
        p_c = P_hat[:, c]
        idx = np.clip(np.digitize(p_c, bins) - 1, 0, n_bins - 1)
        for b in range(n_bins):
            m = idx == b
            if not np.any(m):
                continue
            ece += (m.sum() / N) * abs((y_idx[m] == c).mean() - p_c[m].mean())
    return ece


def ece_ovr_classbalanced(P_hat, y_true, n_bins=15):
    N, C = P_hat.shape
    y_true = np.asarray(y_true)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    eces = []
    for c in range(C):
        p = P_hat[:, c]
        idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
        ece_c = 0.0
        for b in range(n_bins):
            m = idx == b
            if not m.any():
                continue
            ece_c += (m.sum() / N) * abs((y_true[m] == c).mean() - p[m].mean())
        eces.append(ece_c)
    return float(np.mean(eces))


def kfold_indices(N, k=5, shuffle=True, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(N)
    if shuffle:
        rng.shuffle(idx)
    return np.array_split(idx, k)


def probs_with_cv_Q(R, y_true, classes=None, kfold=5, reg=1e-6):
    N = R.shape[0]
    folds = kfold_indices(N, k=kfold, shuffle=True, seed=0)
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    P_hat = np.zeros((N, C), float)

    for val_idx in folds:
        train_idx = np.setdiff1d(np.arange(N), val_idx, assume_unique=True)
        Q, classes = learn_Q_soft(R[train_idx], y_true[train_idx], classes=classes, reg=reg)
        P_hat[val_idx] = class_probs_from_R(R[val_idx], Q)
    return P_hat, classes


# Experiment configuration


In [12]:
experiment_seeds = [42, 1, 2, 3, 6]

config = {
    'num_workers': 25,
    'global_iterations': 200,
    'local_iterations': 1,
    'snapshot_interval': 1,
    'compute_loglik': False,
    'sample_concen': True,
    'ece_bins': 15,
    'ece_kfold': 5,
}

data = data_pca
dim = data.shape[2]
y_true = y.numpy().astype(int)

base_hyperparameter = {
    'alpha': 1.0,
    'gamma': 1.0,
    'mu_0': np.zeros((1, dim)),
    'kappa_0': 1.0,
    'nu_0': float(2.01),
    'Psi_0': np.eye(dim),
}
base_hyperprior = {'alpha_a': 1.0, 'alpha_b': 1.0, 'gamma_a': 5.0, 'gamma_b': 1.0}

if config['num_workers'] > data.shape[0]:
    warnings.warn(
        "Number of workers exceeds number of documents. Some workers will not be assigned any documents."
    )


In [13]:
def build_uncertainty_inputs(final_state, workers, hyperparameter, dim):
    K = len(final_state)
    component_data = {k: np.array([]) for k in range(K)}

    samplers = ray.get([worker.get_sampler.remote() for worker in workers])
    for sampler in samplers:
        for component_id, param in sampler.params_ordered.items():
            if hasattr(param, '_X'):
                component_id = int(component_id)
                component_data[component_id] = (
                    np.concatenate([component_data[component_id], param._X], axis=0)
                    if component_data[component_id].size
                    else param._X
                )

    global_components = {}
    for comp_id in range(K):
        X_comp = component_data[comp_id] if component_data[comp_id].shape[0] > 0 else np.zeros((0, dim))
        global_components[comp_id] = Gaussian(
            X=X_comp,
            kappa_0=hyperparameter['kappa_0'],
            nu_0=hyperparameter['nu_0'],
            mu_0=hyperparameter['mu_0'],
            Psi_0=hyperparameter['Psi_0'],
        )

    mk = np.zeros(K)
    for k in final_state.keys():
        mk[k] = final_state[k]['m_k']

    n_kd_total = np.array([])
    for sampler in samplers:
        n_kd = sampler._n_kd
        if n_kd.shape[0] < K:
            zero_rows = K - n_kd.shape[0]
            n_kd = np.concatenate([n_kd, np.zeros((zero_rows, n_kd.shape[1]))], axis=0)
        elif n_kd.shape[0] > K:
            n_kd = n_kd[:K, :]
        n_kd_total = np.concatenate([n_kd_total, n_kd], axis=1) if n_kd_total.size else n_kd

    return global_components, n_kd_total, mk


def collect_cluster_assignments(workers):
    samplers = ray.get([worker.get_sampler.remote() for worker in workers])
    predicted_labels = []
    for sampler in samplers:
        sampler.create_k_dv(transform=True)
        predicted_labels.append(sampler._k_dv_array)
    return np.concatenate(predicted_labels, axis=0).astype(int)


def run_one_seed(seed, data, y_true, base_hyperparameter, base_hyperprior, config):
    if ray.is_initialized():
        ray.shutdown()

    np.random.seed(seed)
    torch.manual_seed(seed)
    ray.init(ignore_reinit_error=True, include_dashboard=False)

    run_hyperparameter = deepcopy(base_hyperparameter)
    start_time = time.time()

    final_state, final_hyper, history, workers, global_marginal_loglikelihoods, runningtime = run_distributed_hdp(
        data_array=data,
        num_workers=config['num_workers'],
        local_iterations=config['local_iterations'],
        global_iterations=config['global_iterations'],
        snapshot_interval=config['snapshot_interval'],
        compute_loglik=config['compute_loglik'],
        sample_concen=config['sample_concen'],
        hyperparameter=run_hyperparameter,
        hyperprior=deepcopy(base_hyperprior),
        base_seed=seed,
    )
    sampler_runtime = time.time() - start_time

    k_dv_global = collect_cluster_assignments(workers)
    acc, assignments = unsupervised_clustering_accuracy(y_true, k_dv_global)
    dda, fp, fn, health_labels = damage_detection_accuracy(k_dv_global, y_true)

    global_components, n_kd_total, mk = build_uncertainty_inputs(
        final_state=final_state,
        workers=workers,
        hyperparameter=run_hyperparameter,
        dim=data.shape[2],
    )

    BS, BSc, BSS, P_hat, R, Q = brier_from_hdp_state(
        data_groups=data,
        global_components=global_components,
        n_kd_total=n_kd_total,
        mk=mk,
        alpha=final_hyper['alpha'],
        gamma=final_hyper['gamma'],
        y_true=y_true,
        labeled_mask=None,
        classes=None,
    )

    P_hdp, classes = probs_with_cv_Q(R, y_true, kfold=config['ece_kfold'])
    ece_hdp_top = ece_toplabel(P_hdp, y_true, n_bins=config['ece_bins'])
    ece_hdp_ovr = ece_ovr(P_hdp, y_true, n_bins=config['ece_bins'], classes=classes)
    ece_hdp_balanced = ece_ovr_classbalanced(P_hdp, y_true, n_bins=config['ece_bins'])
    total_elapsed_time = time.time() - start_time

    ray.shutdown()

    return {
        'seed': seed,
        'accuracy': acc,
        'damage_detection_accuracy': dda,
        'brier_score': BS,
        'brier_skill_score': BSS,
        'ece_toplabel': ece_hdp_top,
        'ece_ovr': ece_hdp_ovr,
        'ece_balanced': ece_hdp_balanced,
        'runtime_seconds': sampler_runtime,
        'total_elapsed_seconds': total_elapsed_time,
        'num_components': len(final_state),
        'alpha': final_hyper['alpha'],
        'gamma': final_hyper['gamma'],
        'false_positive': fp,
        'false_negative': fn,
        'health_labels': health_labels,
        'assignments': assignments,
    }


# Run five-seed experiment


In [ ]:
seed_results = []

for run_id, seed in enumerate(experiment_seeds, start=1):
    print(f"\n===== Seed {seed} ({run_id}/{len(experiment_seeds)}) =====")
    result = run_one_seed(
        seed=seed,
        data=data,
        y_true=y_true,
        base_hyperparameter=base_hyperparameter,
        base_hyperprior=base_hyperprior,
        config=config,
    )
    seed_results.append(result)
    print(
        f"Seed {seed}: ACC={result['accuracy']:.6f}, "
        f"DDA={result['damage_detection_accuracy']:.6f}, "
        f"Brier={result['brier_score']:.6f}, "
        f"ECE_balanced={result['ece_balanced']:.6f}, "
        f"runtime={result['runtime_seconds']:.2f}s, "
        f"K={result['num_components']}"
    )

results_df = pd.DataFrame(seed_results)
metric_columns = [
    'accuracy',
    'damage_detection_accuracy',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
]
summary_df = pd.DataFrame({
    'mean': results_df[metric_columns].mean(),
    'std': results_df[metric_columns].std(ddof=1),
})

print("\nPer-seed results")
display(results_df[[
    'seed',
    'accuracy',
    'damage_detection_accuracy',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
    'num_components',
    'alpha',
    'gamma',
]])

print("\nMean and sample standard deviation over seeds")
display(summary_df)


In [15]:
if ray.is_initialized():
    ray.shutdown()
